### Imports

In [1]:
import sys
sys.dont_write_bytecode = True


import torch
import numpy as np
import random
import os

def set_seeds(seed_value=42):
    """Sets seeds for reproducibility."""
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)

set_seeds(42) 

import json
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import warnings
import logging
from datetime import datetime
warnings.filterwarnings('ignore')

from model import get_model
from config import CFG
from dataset import get_dataset_class
from transform import get_transforms
from runner import run_baseline, run_lodo

torch.manual_seed(CFG["system"]["seed"])
np.random.seed(CFG["system"]["seed"])

device = CFG["system"]["device"]
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

DS = "PACS"
MODEL_NAME = "resnet18"

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu126 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1126 04:04:51.028000 42300 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Device: cuda
PyTorch: 2.8.0+cu126


### DataLoading

In [2]:
train_transform, test_transform = get_transforms(img_size=224, augment=False, use_imagenet_norm=False)

DatasetClass = get_dataset_class(DS)

ld = DatasetClass(
    data_root=CFG["datasets"][DS]["root"],
    transform=train_transform,
    batch_size=CFG["train"]["batch_size"]
)

print("\nData loaders ready!")


Data loaders ready!


### Logging

In [3]:
dataset_name = DS
base_dir = os.path.join(os.getcwd(), dataset_name)
subdirs = ["logs", "checkpoints", "plots"]

for sub in subdirs:
    os.makedirs(os.path.join(base_dir, sub), exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
log_file = os.path.join(base_dir, "logs", f"train_{timestamp}.log")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(f"{dataset_name}_logger")

logger.info(f"Initialized experiment directories for {dataset_name}")
logger.info(f"Logs: {os.path.join(base_dir, 'logs')}")
logger.info(f"Checkpoints: {os.path.join(base_dir, 'checkpoints')}")
logger.info(f"Plots: {os.path.join(base_dir, 'plots')}")

2025-11-26 04:04:52,391 | INFO | Initialized experiment directories for PACS
2025-11-26 04:04:52,391 | INFO | Logs: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet34_experiments\PACS\logs
2025-11-26 04:04:52,392 | INFO | Checkpoints: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet34_experiments\PACS\checkpoints
2025-11-26 04:04:52,392 | INFO | Plots: d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet34_experiments\PACS\plots


### Setup

In [4]:
domains = CFG["datasets"][DS]["domains"]
loaders = {d: {"train": ld.get_dataloader(d, train=True), "val": ld.get_dataloader(d, train=False)} for d in domains}
ckpt_root = os.path.join(base_dir, "checkpoints")
log_dir = os.path.join(base_dir, "logs")
plots_dir = os.path.join(base_dir, "plots")
os.makedirs(ckpt_root, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)
model_factory = lambda cfg, dataset_key: get_model(cfg,dataset=DS)
optimizer_fn = lambda model: optim.AdamW(model.parameters(), lr=CFG["train"]["lr"], weight_decay=CFG["train"].get("weight_decay", 0.01))
device = CFG["system"]["device"]
epochs = CFG["train"]["epochs"]

{
  "lodo_results": {
    "art_painting": 0.8341463414634146,
    "cartoon": 0.7974413646055437,
    "photo": 0.9580838323353293,
    "sketch": 0.6017811704834606
  },
  "timestamp": "20251004_020611"
}

### Leave One Domain Out

In [5]:
lodo_results, lodo_mean, lodo_summary = run_lodo(
    model_fn=model_factory,
    CFG=CFG,
    logger=logger,
    dataset_key=DS,
    domains=domains,
    loaders=loaders,
    optimizer_fn=optimizer_fn,
    device=device,
    ckpt_root=ckpt_root,
    log_dir=log_dir,
    epochs=epochs
)

2025-11-26 04:04:52,718 | INFO | === LODO: Leaving out domain 'art_painting' ===



=== LODO: Leaving out domain 'art_painting' ===


Evaluating: 100%|██████████| 16/16 [00:03<00:00,  4.82it/s]
2025-11-26 04:05:12,253 | INFO | [art_painting] Epoch 1/10 | Train - Loss: 0.5547, Cls: 0.5465, GRQO: 0.0082, Acc: 0.8154 | Val - Loss: 0.9643, Cls: 0.9633, GRQO: 0.0009, Acc: 0.7368
2025-11-26 04:05:12,335 | INFO | [art_painting] New best val acc: 0.7368


[art_painting] Epoch 1/10 | Train - Loss: 0.5547, Cls: 0.5465, GRQO: 0.0082, Acc: 0.8154 | Val - Loss: 0.9643, Cls: 0.9633, GRQO: 0.0009, Acc: 0.7368
[art_painting] New best val acc: 0.7368


Evaluating: 100%|██████████| 16/16 [00:03<00:00,  5.08it/s]
2025-11-26 04:05:31,435 | INFO | [art_painting] Epoch 2/10 | Train - Loss: 0.1070, Cls: 0.1052, GRQO: 0.0018, Acc: 0.9644 | Val - Loss: 0.4709, Cls: 0.4704, GRQO: 0.0006, Acc: 0.8535
2025-11-26 04:05:31,537 | INFO | [art_painting] New best val acc: 0.8535


...

[sketch] Epoch 8/10 | Train - Loss: 0.0090, Cls: 0.0096, GRQO: -0.0006, Acc: 0.9972 | Val - Loss: 1.7539, Cls: 1.7533, GRQO: 0.0006, Acc: 0.6276


Evaluating: 100%|██████████| 31/31 [00:06<00:00,  4.62it/s]
2025-11-26 04:17:41,203 | INFO | [sketch] Epoch 9/10 | Train - Loss: 0.0087, Cls: 0.0095, GRQO: -0.0008, Acc: 0.9977 | Val - Loss: 1.2792, Cls: 1.2792, GRQO: 0.0001, Acc: 0.6709


[sketch] Epoch 9/10 | Train - Loss: 0.0087, Cls: 0.0095, GRQO: -0.0008, Acc: 0.9977 | Val - Loss: 1.2792, Cls: 1.2792, GRQO: 0.0001, Acc: 0.6709


Evaluating: 100%|██████████| 31/31 [00:07<00:00,  4.32it/s]
2025-11-26 04:18:02,494 | INFO | [sketch] Epoch 10/10 | Train - Loss: 0.0245, Cls: 0.0253, GRQO: -0.0008, Acc: 0.9918 | Val - Loss: 1.2794, Cls: 1.2786, GRQO: 0.0008, Acc: 0.6541
2025-11-26 04:18:02,494 | INFO | [sketch] Best Acc: 0.6897
2025-11-26 04:18:02,494 | INFO | ------------------------------------------------------------
2025-11-26 04:18:02,494 | INFO | LODO finished | Mean Acc: 0.8156 | Summary saved to d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet34_experiments\PACS\logs\lodo_summary_20251126_041802.json


[sketch] Epoch 10/10 | Train - Loss: 0.0245, Cls: 0.0253, GRQO: -0.0008, Acc: 0.9918 | Val - Loss: 1.2794, Cls: 1.2786, GRQO: 0.0008, Acc: 0.6541
[sketch] Best Acc: 0.6897
------------------------------------------------------------
LODO finished | Mean Acc: 0.8156
Summary saved to d:\Haseeb\SPROJ\GRQO\Vit-GRQO\resnet34_experiments\PACS\logs\lodo_summary_20251126_041802.json


### Baseline

In [6]:
baseline_results, baseline_mean = run_baseline(
    model_name=MODEL_NAME,
    CFG=CFG,
    logger=logger,
    dataset_key=DS,
    domains=domains,
    loaders=loaders,
    optimizer_fn=optimizer_fn,
    device=device,
    epochs=CFG["train"]["epochs"]
)

2025-11-26 04:18:02,505 | INFO | Initializing ResNet baseline: resnet18
2025-11-26 04:18:02,586 | INFO | === Baseline LODO: Leaving out domain 'art_painting' ===


Initializing ResNet baseline: resnet18

=== Baseline LODO: Leaving out domain 'art_painting' ===


2025-11-26 04:18:18,104 | INFO | [art_painting] Epoch 1/10 | Train - Loss: 0.5019, Acc: 0.8341 | Val Acc: 0.6406


[art_painting] Epoch 1/10 | Train - Loss: 0.5019, Acc: 0.8341 | Val Acc: 0.6406


2025-11-26 04:18:33,903 | INFO | [art_painting] Epoch 2/10 | Train - Loss: 0.0956, Acc: 0.9767 | Val Acc: 0.7231


...

[sketch] Epoch 8/10 | Train - Loss: 0.0019, Acc: 1.0000 | Val Acc: 0.6215


2025-11-26 04:28:20,636 | INFO | [sketch] Epoch 9/10 | Train - Loss: 0.0017, Acc: 1.0000 | Val Acc: 0.6040


[sketch] Epoch 9/10 | Train - Loss: 0.0017, Acc: 1.0000 | Val Acc: 0.6040


2025-11-26 04:28:37,627 | INFO | [sketch] Epoch 10/10 | Train - Loss: 0.0013, Acc: 1.0000 | Val Acc: 0.6093
2025-11-26 04:28:37,627 | INFO | [sketch] Best Val Acc: 0.6215
2025-11-26 04:28:37,627 | INFO | ------------------------------------------------------------
2025-11-26 04:28:37,627 | INFO | Baseline LODO (resnet18) finished | Mean Acc: 0.7724


[sketch] Epoch 10/10 | Train - Loss: 0.0013, Acc: 1.0000 | Val Acc: 0.6093
[sketch] Best Val Acc: 0.6215
------------------------------------------------------------
Baseline LODO (resnet18) finished | Mean Acc: 0.7724
